In [ ]:
# Cell 1
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

In [ ]:
# Cell 2
# Validation settings
n_splits = 5
seed = 10

kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)


In [ ]:
# Cell 3
# Data preparation
def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    # Convert boolean features to numeric
    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    # Dummy encoding and align columns
    X_train = pd.get_dummies(X_train, drop_first=False)
    X_val = pd.get_dummies(X_val, drop_first=False)

    X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

    return X_train, X_val

In [ ]:
# Cell 4
# Evaluate one parameter set
def evaluate_xgb_params(prefix, params, upsample=True):
    X = pd.read_csv(f"data/{prefix}_X_train.csv", keep_default_na=False)
    y = pd.read_csv(f"data/{prefix}_y_train.csv", keep_default_na=False).values.ravel()

    cv_scores = {
        "fold": [],
        "test_precision": [],
        "test_recall": [],
        "test_roc_auc": [],
        "test_accuracy": [],
        "test_f1": []
    }

    splits = list(kf.split(X, y))

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_train = X.iloc[train_idx].copy()
        y_train = y[train_idx].copy()

        X_val = X.iloc[val_idx].copy()
        y_val = y[val_idx].copy()

        # Apply oversampling to training data only
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(X_train, y_train)

            if not isinstance(X_train, pd.DataFrame):
                X_train = pd.DataFrame(X_train, columns=X.columns)

        X_train_prepared, X_val_prepared = prepare_fold_data(X_train, X_val)

        clf = XGBClassifier(
            n_estimators=250,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=seed,
            tree_method="hist",
            **params
        )

        clf.fit(X_train_prepared, y_train)

        y_pred = clf.predict(X_val_prepared)
        y_prob = clf.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx + 1)
        cv_scores["test_precision"].append(precision_score(y_val, y_pred))
        cv_scores["test_recall"].append(recall_score(y_val, y_pred))
        cv_scores["test_roc_auc"].append(roc_auc_score(y_val, y_prob))
        cv_scores["test_accuracy"].append(accuracy_score(y_val, y_pred))
        cv_scores["test_f1"].append(f1_score(y_val, y_pred))

    # Convert to arrays for aggregation
    for key in ["test_precision", "test_recall", "test_roc_auc", "test_accuracy", "test_f1"]:
        cv_scores[key] = np.array(cv_scores[key])

    # Return mean performance for this parameter set
    summary = {
        "params": params,
        "precision_mean": cv_scores["test_precision"].mean(),
        "recall_mean": cv_scores["test_recall"].mean(),
        "roc_auc_mean": cv_scores["test_roc_auc"].mean(),
        "accuracy_mean": cv_scores["test_accuracy"].mean(),
        "f1_mean": cv_scores["test_f1"].mean()
    }

    return cv_scores, summary

In [ ]:
# Cell 5
# Hyperparameter grid
xgb_param_grid = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5, 7],
    "min_child_weight": [1, 5],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.6, 0.8],
    "gamma": [0.0, 0.1, 1.0],
    "reg_lambda": [0.1, 1.0]
}

In [ ]:
# Cell 6
# Grid search for best parameters
def tune_xgboost(prefix, upsample=True):
    all_results = []

    for params in ParameterGrid(xgb_param_grid):
        _, summary = evaluate_xgb_params(prefix, params, upsample=upsample)
        all_results.append(summary)

    results_df = pd.DataFrame(all_results)
    # Rank models by ROC AUC, then F1, then Accuracy
    results_df = results_df.sort_values(
        by=["roc_auc_mean", "f1_mean", "accuracy_mean"],
        ascending=False
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]
    
    best_params = best_row["params"]

    print(f"Best params for {prefix}:")
    print(best_params)
    print("Best mean ROC AUC:", round(best_row["roc_auc_mean"], 3))
    print("Best mean precision:", round(best_row["precision_mean"], 3))
    print("Best mean recall:", round(best_row["recall_mean"], 3))
    print("Best mean accuracy:", round(best_row["accuracy_mean"], 3))
    print("Best mean F1:", round(best_row["f1_mean"], 3))

    return results_df, best_params


In [ ]:
# Cell 7
# Evaluate best model with CV + SE
def cross_validate_best_xgb(prefix, best_params, upsample=True):
    cv_scores, _ = evaluate_xgb_params(prefix, best_params, upsample=upsample)

    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["test_precision"],
        "recall": cv_scores["test_recall"],
        "roc_auc": cv_scores["test_roc_auc"],
        "accuracy": cv_scores["test_accuracy"],
        "f1": cv_scores["test_f1"]
    })

    # Mean performance
    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])

    # Cross-validated SE
    n = n_splits

    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(n),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(n),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(n),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(n),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(n)
    }])

    summary_df = pd.concat([summary_df, mean_row, se_row], ignore_index=True)

    print(f"Results for {prefix} (upsample={upsample})")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
        se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
        print(f"Mean {metric}: {mean_value:.3f} ± {se_value:.3f}")

    return cv_scores, summary_df

In [ ]:
# Cell 8
# Run for all tasks
xgb_results_mask_before, best_xgb_params_mask_before = tune_xgboost(
    "mask_before",
    upsample=True
)
cv_scores_xgb_mask_before, summary_xgb_mask_before = cross_validate_best_xgb(
    "mask_before",
    best_xgb_params_mask_before,
    upsample=True
)

xgb_results_mask_after, best_xgb_params_mask_after = tune_xgboost(
    "mask_after",
    upsample=True
)
cv_scores_xgb_mask_after, summary_xgb_mask_after = cross_validate_best_xgb(
    "mask_after",
    best_xgb_params_mask_after,
    upsample=True
)

xgb_results_general_before, best_xgb_params_general_before = tune_xgboost(
    "general_before",
    upsample=True
)
cv_scores_xgb_general_before, summary_xgb_general_before = cross_validate_best_xgb(
    "general_before",
    best_xgb_params_general_before,
    upsample=True
)

xgb_results_general_after, best_xgb_params_general_after = tune_xgboost(
    "general_after",
    upsample=True
)
cv_scores_xgb_general_after, summary_xgb_general_after = cross_validate_best_xgb(
    "general_after",
    best_xgb_params_general_after,
    upsample=True
)

Best params for mask_before:
{'colsample_bytree': 0.6, 'gamma': 0.1, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 1, 'reg_lambda': 1.0, 'subsample': 0.8}
Best mean ROC AUC: 0.861
Best mean precision: 0.604
Best mean recall: 0.696
Best mean accuracy: 0.803
Best mean F1: 0.647
Results for mask_before (upsample=True)
Mean precision: 0.604 ± 0.007
Mean recall: 0.696 ± 0.003
Mean roc_auc: 0.861 ± 0.003
Mean accuracy: 0.803 ± 0.003
Mean f1: 0.647 ± 0.003
Best params for mask_after:
{'colsample_bytree': 0.6, 'gamma': 1.0, 'learning_rate': 0.05, 'max_depth': 7, 'min_child_weight': 5, 'reg_lambda': 1.0, 'subsample': 0.8}
Best mean ROC AUC: 0.892
Best mean precision: 0.898
Best mean recall: 0.88
Best mean accuracy: 0.844
Best mean F1: 0.889
Results for mask_after (upsample=True)
Mean precision: 0.898 ± 0.001
Mean recall: 0.880 ± 0.004
Mean roc_auc: 0.892 ± 0.002
Mean accuracy: 0.844 ± 0.002
Mean f1: 0.889 ± 0.002
Best params for general_before:
{'colsample_bytree': 0.6, 'gamma': 1.

In [ ]:
# Cell 9
# Build comparison table
def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
    se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
    return mean_value, se_value


summaries = {
    "mask_before": summary_xgb_mask_before,
    "mask_after": summary_xgb_mask_after,
    "general_before": summary_xgb_general_before,
    "general_after": summary_xgb_general_after
}

rows = []

for model_name, summary in summaries.items():
    row = {"model": model_name}
    
    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary, metric)
        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"
    
    rows.append(row)

comparison_df = pd.DataFrame(rows)

comparison_df

,model,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,mask_before,0.604055,0.006854,0.604 ± 0.007,0.696025,0.002772,0.696 ± 0.003,0.861052,0.002565,0.861 ± 0.003,0.802721,0.003205,0.803 ± 0.003,0.646665,0.003439,0.647 ± 0.003
1,mask_after,0.897564,0.001485,0.898 ± 0.001,0.880084,0.003505,0.880 ± 0.004,0.892109,0.002170,0.892 ± 0.002,0.844236,0.002361,0.844 ± 0.002,0.888720,0.001849,0.889 ± 0.002
2,general_before,0.731507,0.004420,0.732 ± 0.004,0.731104,0.006372,0.731 ± 0.006,0.792011,0.003350,0.792 ± 0.003,0.715169,0.004353,0.715 ± 0.004,0.731246,0.004379,0.731 ± 0.004
3,general_after,0.885504,0.001036,0.886 ± 0.001,0.798361,0.002177,0.798 ± 0.002,0.849352,0.000510,0.849 ± 0.001,0.779070,0.000794,0.779 ± 0.001,0.839666,0.000827,0.840 ± 0.001


In [ ]:
# Cell 10
pd.DataFrame(comparison_df).to_csv("results/xgb_mask_before.csv", index=False)